In [18]:
#importar librerias
import json
import copy
import matplotlib.pyplot as plt
import numpy as np

In [90]:
#función para generar gráficas
def graficar_(experimentos, resultados, guardar, ver=False):
    for i in range(len(experimentos)):
        try:
            with open(experimentos[i], "r", encoding="utf-8") as a:
                base = json.load(a)
        except:
            print("NO SE ENCONTRÓ ", experimentos[i])
            continue

        try:
            with open(resultados[i], "r", encoding="utf-8") as f:
                res = json.load(f)
        except:
            print("NO SE ENCONTRÓ ", resultados[i])
            continue

        fig, axs = plt.subplots(2, 2, figsize=(10, 8))
        

        #---------------------------------------------------------
        axs[0, 0].plot([i for i in range(len(res["Best loss"]))], res["Best loss"], label=f"Mejor loss ({min(res["Best loss"])})")
        axs[0, 0].plot([i for i in range(len(res["Worst loss"]))], res["Worst loss"], label=f"Peor loss ({max(res["Worst loss"])})")
        axs[0, 0].set_xlabel("Iteraciones")
        axs[0, 0].set_ylabel("Loss ("+base["Fitness"]+")")
        axs[0, 0].set_title("Loss obtenido en "+resultados[i])
        axs[0, 0].grid(True)
        axs[0, 0].legend(loc="upper right")

        #---------------------------------------------------------
        axs[0, 1].plot([i for i in range(len(res["Best accuracy"]))], res["Best accuracy"], label=f"Mejor accuracy ({max(res["Best accuracy"])})")
        axs[0, 1].plot([i for i in range(len(res["Worst accuracy"]))], res["Worst accuracy"], label=f"Peor accuracy ({min(res["Worst accuracy"])})")
        axs[0, 1].set_xlabel("Iteraciones")
        axs[0, 1].set_ylabel("Accuracy")
        axs[0, 1].set_title("Accuracy obtenido en "+resultados[i])
        axs[0, 1].grid(True)
        axs[0, 1].legend(loc="upper right")

        #---------------------------------------------------------
        superior = np.array(res["Promedio"]) + np.array(res["Desvest"])
        inferior = np.array(res["Promedio"]) - np.array(res["Desvest"])
        axs[1, 0].plot([i for i in range(len(res["Best iteration"]))], res["Best iteration"], label=f"Mejor fitness ({min(res["Best iteration"])})")
        axs[1, 0].plot([i for i in range(len(res["Worst iteration"]))], res["Worst iteration"], label=f"Peor fitness ({max(res["Worst iteration"])})")
        axs[1, 0].plot([i for i in range(len(res["Promedio"]))], res["Promedio"], linewidth=1.5, color="teal", label="Valor promedio\npor iteracion")                # línea fuerte
        axs[1, 0].plot([i for i in range(len(superior))], superior, linestyle='--', linewidth=1.5, alpha=0.5, color="teal", label="Percentiles\n25 y 75")     # borde inferior
        axs[1, 0].plot([i for i in range(len(inferior))], inferior, linestyle='--', linewidth=1.5, alpha=0.5, color="teal")     # borde superior
        axs[1, 0].fill_between([i for i in range(len(superior))], superior, inferior, alpha=0.2, color="teal", label="±1σ")
        axs[1, 0].set_xlabel("Iteraciones")
        axs[1, 0].set_ylabel("Fitness")
        axs[1, 0].set_title("Fitness obtenido en "+resultados[i])
        axs[1, 0].grid(True)
        axs[1, 0].legend(loc="upper right")

        #---------------------------------------------------------
        axs[1, 1].plot([i for i in range(len(res["Succes"]))], res["Succes"], label="Succes")
        axs[1, 1].set_xlabel("Iteraciones")
        axs[1, 1].set_ylabel("Succes")
        axs[1, 1].set_title("Succes (si o no, 1 o 0)")
        axs[1, 1].grid()
        axs[1, 1].legend(loc="upper right")

        #---------------------------------------------------------
        prom=np.mean(np.array(res["Tiempo por iteración"]))
        tot=sum(np.array(res["Tiempo por iteración"]))
        fig.suptitle(f"Resultados para "+resultados[i]+f"\n{prom:.2f} segundos por iteracion\n{tot:.2f} segundos en total\npruebas en datos de: \""+base["Reporte"]+"\"")
        plt.tight_layout()
        plt.savefig(guardar[i], dpi=300)
        if(ver==True):
            plt.show()
        else:
            plt.close()

def graficar(ruta, ver=False):
    with open(ruta, "r", encoding="utf-8") as a:
        base = json.load(a)

    experimentos=[]
    resultados=[]
    graficas=[]

    for i in range(base["N archivos"]):
        with open(base["Archivos"][i], "r", encoding="utf-8") as f:
                archvio=json.load(f)

        experimentos.append(base["Archivos"][i])
        resultados.append(archvio["Archivo salida"])
        graficas.append(archvio["Archivo graficas"])

    graficar_(experimentos, 
              resultados, 
              graficas,
              ver)

if __name__=="__main__":
    graficar("../j_7.json")

In [88]:
#crear los jsons de los experimentos
seed_base=277010
dataset="mini_box"
optimizador="SHADE"

Fs=[[0.75 for i in range(10)],
    [0.8  for i in range(10)]]

Crs=[[0.8  for i in range(10)],
     [0.9  for i in range(10)]]

extras=[{"Data type":        "double",
         "Dataset":          dataset, 
         "N iters":          1000, 
         "N inds":           200, 
         "N experimentos":   1, 
         "Seed":             seed_base+i%10, 
         "Archivo salida":   "", #--------------------------------------
         "Archivo graficas": "", #--------------------------------------
         "Optimizador":      optimizador,
         "Lower bound":      -1.0, 
         "Upper bound":      1.0, 
         "Fitness":          "accuracy", 
         "F":                0.5, #--------------------------------------
         "C_r":              0.5, #--------------------------------------
         "N capas":          4,
         "Estructura":       [2, 5, 3, 1],
         "Activaciones":     ["tanh", "tanh", "tanh", "sigmoid"],
         "Verbose":          -1, 
         "Reporte":          "train",
         "Clip":             1, 
         "Normalizacion":    1,
         "Mutacion":         "pbest"}
         for i in range(len(Fs)*len(Fs[0]))]
     
count=0
for i in range(len(Fs)):
    for j in range(len(Fs[i])):
        extras[count]["Archivo salida"]="../resultados/"+dataset+f"_{i}_{j}_"+extras[count]["Fitness"]+"_"+optimizador+".txt"
        extras[count]["Archivo graficas"]="../graficas/"+dataset+f"_{i}_{j}_"+extras[count]["Fitness"]+"_"+optimizador+".png"

        extras[count]["F"]=Fs[i][j]
        extras[count]["C_r"]=Crs[i][j]

        print(count, 
              "../resultados/"+dataset+f"_{i}_{j}_"+extras[count]["Fitness"]+"_"+optimizador+".txt", 
              "../graficas/"+dataset+f"_{i}_{j}_"+extras[count]["Fitness"]+"_"+optimizador+".png", 
              Fs[i][j], 
              Crs[i][j])

        count+=1

count=0
nombre_archivo=[]
for i in range(len(Fs)):
    for j in range(len(Fs[i])):
        nombre_archivo.append(f"../experimentos/j_{i}_{j}_"+extras[count]["Fitness"]+"_"+dataset+"_"+optimizador+".json")
        count+=1

for i, nuevo_json in enumerate(extras):
    with open(nombre_archivo[i], "w", encoding="utf-8") as f:
        json.dump(nuevo_json, f, indent=4, ensure_ascii=False)


iguales=[[], [], [], [], [], []]
count=0
for i in range(len(Fs)):
    for j in range(len(Fs[i])):
        iguales[i].append(f"../experimentos/j_{i}_{j}_"+extras[count]["Fitness"]+"_"+dataset+"_"+optimizador+".json")
        count+=1

data = {
    "N archivos": len(nombre_archivo),
    "N hilos":    7,
    "Archivos":   nombre_archivo,
    "Iguales":    iguales
}

# Writing to sample.json
with open("../j_7_.json", "w") as h:
    json.dump(data, h, indent=4)

0 ../resultados/mini_box_0_0_accuracy_SHADE.txt ../graficas/mini_box_0_0_accuracy_SHADE.png 0.75 0.8
1 ../resultados/mini_box_0_1_accuracy_SHADE.txt ../graficas/mini_box_0_1_accuracy_SHADE.png 0.75 0.8
2 ../resultados/mini_box_0_2_accuracy_SHADE.txt ../graficas/mini_box_0_2_accuracy_SHADE.png 0.75 0.8
3 ../resultados/mini_box_0_3_accuracy_SHADE.txt ../graficas/mini_box_0_3_accuracy_SHADE.png 0.75 0.8
4 ../resultados/mini_box_0_4_accuracy_SHADE.txt ../graficas/mini_box_0_4_accuracy_SHADE.png 0.75 0.8
5 ../resultados/mini_box_0_5_accuracy_SHADE.txt ../graficas/mini_box_0_5_accuracy_SHADE.png 0.75 0.8
6 ../resultados/mini_box_0_6_accuracy_SHADE.txt ../graficas/mini_box_0_6_accuracy_SHADE.png 0.75 0.8
7 ../resultados/mini_box_0_7_accuracy_SHADE.txt ../graficas/mini_box_0_7_accuracy_SHADE.png 0.75 0.8
8 ../resultados/mini_box_0_8_accuracy_SHADE.txt ../graficas/mini_box_0_8_accuracy_SHADE.png 0.75 0.8
9 ../resultados/mini_box_0_9_accuracy_SHADE.txt ../graficas/mini_box_0_9_accuracy_SHADE.png

In [ ]:
#crear jsons
with open("./experimentos/j.json", "r", encoding="utf-8") as f:
    base = json.load(f)

# 2. Definir los cambios que quieres hacer
criterios=[{"Archivo salida": "../resultados/micro_mnist_0_0.txt", "Archivo graficas":"../graficas/micro_mnist_0_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"micro_mnist"}, 
           {"Archivo salida": "../resultados/micro_mnist_0_1.txt", "Archivo graficas":"../graficas/micro_mnist_0_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"micro_mnist"}, 
           {"Archivo salida": "../resultados/micro_mnist_0_2.txt", "Archivo graficas":"../graficas/micro_mnist_0_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"micro_mnist"}, 
           {"Archivo salida": "../resultados/micro_mnist_0_3.txt", "Archivo graficas":"../graficas/micro_mnist_0_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"micro_mnist"}, 
           {"Archivo salida": "../resultados/micro_mnist_0_4.txt", "Archivo graficas":"../graficas/micro_mnist_0_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_0_5.txt", "Archivo graficas":"../graficas/micro_mnist_0_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"micro_mnist"},
           
           {"Archivo salida": "../resultados/micro_mnist_1_0.txt", "Archivo graficas":"../graficas/micro_mnist_1_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_1_1.txt", "Archivo graficas":"../graficas/micro_mnist_1_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_1_2.txt", "Archivo graficas":"../graficas/micro_mnist_1_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_1_3.txt", "Archivo graficas":"../graficas/micro_mnist_1_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_1_4.txt", "Archivo graficas":"../graficas/micro_mnist_1_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_1_5.txt", "Archivo graficas":"../graficas/micro_mnist_1_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"micro_mnist"},
           
           {"Archivo salida": "../resultados/micro_mnist_2_0.txt", "Archivo graficas":"../graficas/micro_mnist_2_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_2_1.txt", "Archivo graficas":"../graficas/micro_mnist_2_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_2_2.txt", "Archivo graficas":"../graficas/micro_mnist_2_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_2_3.txt", "Archivo graficas":"../graficas/micro_mnist_2_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_2_4.txt", "Archivo graficas":"../graficas/micro_mnist_2_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_2_5.txt", "Archivo graficas":"../graficas/micro_mnist_2_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"micro_mnist"},

           {"Archivo salida": "../resultados/micro_mnist_3_0.txt", "Archivo graficas":"../graficas/micro_mnist_3_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_3_1.txt", "Archivo graficas":"../graficas/micro_mnist_3_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_3_2.txt", "Archivo graficas":"../graficas/micro_mnist_3_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_3_3.txt", "Archivo graficas":"../graficas/micro_mnist_3_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_3_4.txt", "Archivo graficas":"../graficas/micro_mnist_3_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_3_5.txt", "Archivo graficas":"../graficas/micro_mnist_3_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"micro_mnist"},

           {"Archivo salida": "../resultados/micro_mnist_4_0.txt", "Archivo graficas":"../graficas/micro_mnist_4_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_4_1.txt", "Archivo graficas":"../graficas/micro_mnist_4_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_4_2.txt", "Archivo graficas":"../graficas/micro_mnist_4_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_4_3.txt", "Archivo graficas":"../graficas/micro_mnist_4_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_4_4.txt", "Archivo graficas":"../graficas/micro_mnist_4_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_4_5.txt", "Archivo graficas":"../graficas/micro_mnist_4_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"micro_mnist"},

           {"Archivo salida": "../resultados/micro_mnist_5_0.txt", "Archivo graficas":"../graficas/micro_mnist_5_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_5_1.txt", "Archivo graficas":"../graficas/micro_mnist_5_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_5_2.txt", "Archivo graficas":"../graficas/micro_mnist_5_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_5_3.txt", "Archivo graficas":"../graficas/micro_mnist_5_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_5_4.txt", "Archivo graficas":"../graficas/micro_mnist_5_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"micro_mnist"},
           {"Archivo salida": "../resultados/micro_mnist_5_5.txt", "Archivo graficas":"../graficas/micro_mnist_5_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"micro_mnist"},
           






           
           {"Archivo salida": "../resultados/mini_box_0_0.txt", "Archivo graficas":"../graficas/mini_box_0_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"mini_box"}, 
           {"Archivo salida": "../resultados/mini_box_0_1.txt", "Archivo graficas":"../graficas/mini_box_0_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"mini_box"}, 
           {"Archivo salida": "../resultados/mini_box_0_2.txt", "Archivo graficas":"../graficas/mini_box_0_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"mini_box"}, 
           {"Archivo salida": "../resultados/mini_box_0_3.txt", "Archivo graficas":"../graficas/mini_box_0_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"mini_box"}, 
           {"Archivo salida": "../resultados/mini_box_0_4.txt", "Archivo graficas":"../graficas/mini_box_0_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_0_5.txt", "Archivo graficas":"../graficas/mini_box_0_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.5, "C_r": 0.5,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"mini_box"},
           
           {"Archivo salida": "../resultados/mini_box_1_0.txt", "Archivo graficas":"../graficas/mini_box_1_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_1_1.txt", "Archivo graficas":"../graficas/mini_box_1_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_1_2.txt", "Archivo graficas":"../graficas/mini_box_1_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_1_3.txt", "Archivo graficas":"../graficas/mini_box_1_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_1_4.txt", "Archivo graficas":"../graficas/mini_box_1_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_1_5.txt", "Archivo graficas":"../graficas/mini_box_1_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.6, "C_r": 0.62,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"mini_box"},
           
           {"Archivo salida": "../resultados/mini_box_2_0.txt", "Archivo graficas":"../graficas/mini_box_2_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_2_1.txt", "Archivo graficas":"../graficas/mini_box_2_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_2_2.txt", "Archivo graficas":"../graficas/mini_box_2_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_2_3.txt", "Archivo graficas":"../graficas/mini_box_2_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_2_4.txt", "Archivo graficas":"../graficas/mini_box_2_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_2_5.txt", "Archivo graficas":"../graficas/mini_box_2_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.65, "C_r": 0.68,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"mini_box"},

           {"Archivo salida": "../resultados/mini_box_3_0.txt", "Archivo graficas":"../graficas/mini_box_3_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_3_1.txt", "Archivo graficas":"../graficas/mini_box_3_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_3_2.txt", "Archivo graficas":"../graficas/mini_box_3_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_3_3.txt", "Archivo graficas":"../graficas/mini_box_3_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_3_4.txt", "Archivo graficas":"../graficas/mini_box_3_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_3_5.txt", "Archivo graficas":"../graficas/mini_box_3_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.7, "C_r": 0.74,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"mini_box"},

           {"Archivo salida": "../resultados/mini_box_4_0.txt", "Archivo graficas":"../graficas/mini_box_4_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_4_1.txt", "Archivo graficas":"../graficas/mini_box_4_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_4_2.txt", "Archivo graficas":"../graficas/mini_box_4_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_4_3.txt", "Archivo graficas":"../graficas/mini_box_4_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_4_4.txt", "Archivo graficas":"../graficas/mini_box_4_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_4_5.txt", "Archivo graficas":"../graficas/mini_box_4_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.75, "C_r": 0.8,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"mini_box"},

           {"Archivo salida": "../resultados/mini_box_5_0.txt", "Archivo graficas":"../graficas/mini_box_5_0.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277010, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_5_1.txt", "Archivo graficas":"../graficas/mini_box_5_1.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277011, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_5_2.txt", "Archivo graficas":"../graficas/mini_box_5_2.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277012, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_5_3.txt", "Archivo graficas":"../graficas/mini_box_5_3.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277013, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_5_4.txt", "Archivo graficas":"../graficas/mini_box_5_4.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277014, "N experimentos":1, "Dataset":"mini_box"},
           {"Archivo salida": "../resultados/mini_box_5_5.txt", "Archivo graficas":"../graficas/mini_box_5_5.png", "Lower bound": -1.0, "Upper bound": 1.0, "F": 0.8, "C_r": 0.9,  "N iters": 1000, "N inds": 200, "Verbose": -1, "Seed":277015, "N experimentos":1, "Dataset":"mini_box"}]

lista_todos_archivos=[]
nombre_archivo=[]
for i in range()

# 3. Crear copias modificadas
for i, criterio in enumerate(criterios):
    nuevo_json = copy.deepcopy(base)

    # Cambiar campos específicos
    nuevo_json["Archivo salida"] = criterio["Archivo salida"]
    nuevo_json["Archivo graficas"] = criterio["Archivo graficas"]

    nuevo_json["Lower bound"] = criterio["Lower bound"]
    nuevo_json["Upper bound"] = criterio["Upper bound"]

    nuevo_json["F"] = criterio["F"]
    nuevo_json["C_r"] = criterio["C_r"]

    nuevo_json["N iters"] = criterio["N iters"]
    nuevo_json["N inds"] = criterio["N inds"]
    nuevo_json["Verbose"] = criterio["Verbose"]

    nuevo_json["Seed"] = criterio["Seed"]
    nuevo_json["Dataset"] = criterio["Dataset"]
    nuevo_json["N experimentos"] = criterio["N experimentos"]

    # 4. Guardar nuevo archivo
    nombre_archivo = 
    lista_todos_archivos.append(nombre_archivo)

    with open(nombre_archivo, "w", encoding="utf-8") as f:
        json.dump(nuevo_json, f, indent=4, ensure_ascii=False)